# Worker 2 · Pemantauan Penyuntingan Wikipedia (Soal 1c)
**Big Data** · dijalankan di **PC Worker** (latihan solo: dua notebook Worker di JupyterLab yang sama)

Worker bergabung ke consumer group `wiki-pekerja`, membaca partisi bagiannya dari `wiki.recentchange.raw`,
mengolah tiap batch dengan **Spark lokal**, mengirim hasil ke `wiki.hasil`, lalu **commit offset setelah hasil terkirim**
(at-least-once).

Jalankan Sel 1 → 3. **Sel 3 terus berjalan**; hentikan dengan Interrupt (tombol ■) saat uji decoupling.

## Sel 1. Konfigurasi dan producer laporan

In [1]:
# ===== SEL 1: Konfigurasi, producer hasil/status =====
import os, sys, json, time, socket
from confluent_kafka import Producer, Consumer, KafkaError, KafkaException, TopicPartition

# ---------------- ISI SESUAI LINGKUNGAN ----------------
NOMOR_WORKER = 3                 # solo: 1 dan 2 | kelompok: 1 ... 4
MASTER_IP    = "192.168.1.15"       # kelompok: IP PC Master (sama dengan advertised.listeners Kafka)
PAKAI_SPARK  = True              # False = olah dengan Python biasa (cadangan jika Spark lokal bermasalah)
SIMULASI_CRASH_SEBELUM_COMMIT = False   # True = kirim hasil 1 batch lalu berhenti TANPA commit (uji duplikat)
# -------------------------------------------------------

BOOTSTRAP    = f"{MASTER_IP}:9092"
TOPIC_RAW    = "wiki.recentchange.raw"
TOPIC_HASIL  = "wiki.hasil"
TOPIC_STATUS = "wiki.status"
GROUP_ID     = "wiki-pekerja"
HOST         = f"{socket.gethostname()}-w{NOMOR_WORKER}"

BATCH_DETIK  = 10                # satu batch = pesan yang terkumpul selama 10 detik ...
BATCH_MAKS   = 2000              # ... atau maksimum 2000 pesan
DETAK_DETIK  = 5                 # interval laporan HIDUP ke Master

produser = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
    "linger.ms": 50,
    "compression.type": "lz4",
})

galat_kirim = []
def _hasil_kirim(err, msg):
    if err is not None:
        galat_kirim.append(str(err))

def kirim(topic, key, obj):
    while True:
        try:
            produser.produce(topic, key=key, value=json.dumps(obj, ensure_ascii=False).encode("utf-8"),
                             on_delivery=_hasil_kirim)
            break
        except BufferError:
            produser.poll(0.5)
    produser.poll(0)

def lapor(status, **tambahan):
    kirim(TOPIC_STATUS, HOST, {"worker": HOST, "nomor": NOMOR_WORKER, "status": status,
                               "waktu": time.time(), **tambahan})

meta = produser.list_topics(timeout=5)
for t in (TOPIC_RAW, TOPIC_HASIL, TOPIC_STATUS):
    ada = t in meta.topics
    print(f"{t:<24}", f"{len(meta.topics[t].partitions)} partisi" if ada else "BELUM ADA -> jalankan Sel 1 Master dulu")
print("Nama Worker:", HOST)

wiki.recentchange.raw    4 partisi
wiki.hasil               3 partisi
wiki.status              1 partisi
Nama Worker: NthaanSnn-w3


## Sel 2. Spark lokal dan fungsi pengolahan batch
Sesuai pola materi: Worker mengolah datanya sendiri dengan Spark lokal (`local[2]`). Setiap pesan mentah menghasilkan **satu** rekaman hasil yang membawa `partition` dan `offset` asalnya, sehingga Master bisa memeriksa apakah ada pesan yang hilang atau ganda.

In [2]:
# ===== SEL 2: Spark lokal, skema, dan pengolahan batch =====
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               IntegerType, BooleanType, DoubleType)

SKEMA = StructType([
    StructField("id", LongType()),         StructField("type", StringType()),
    StructField("wiki", StringType()),     StructField("server_name", StringType()),
    StructField("title", StringType()),    StructField("namespace", IntegerType()),
    StructField("user", StringType()),     StructField("is_anon", BooleanType()),
    StructField("anon_kind", StringType()), StructField("bot", BooleanType()),
    StructField("minor", BooleanType()),   StructField("timestamp", LongType()),
    StructField("length_old", LongType()), StructField("length_new", LongType()),
    StructField("ingest_ts", DoubleType()),
])
KOLOM = [f.name for f in SKEMA.fields]
SKEMA_MENTAH = StructType([StructField("partition", IntegerType()), StructField("offset", LongType()),
                            StructField("value", StringType())])

spark = None
if PAKAI_SPARK:
    spark = (SparkSession.builder
             .appName(f"wiki-worker{NOMOR_WORKER}")
             .master("local[2]")
             .config("spark.driver.memory", "1g")
             .config("spark.driver.host", "127.0.0.1")
             .config("spark.driver.bindAddress", "127.0.0.1")
             .config("spark.sql.shuffle.partitions", "4")
             .config("spark.pyspark.python", sys.executable)
             .config("spark.pyspark.driver.python", sys.executable)
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark", spark.version, "| master:", spark.sparkContext.master, "| UI:", spark.sparkContext.uiWebUrl)
else:
    print("Mode Python biasa (PAKAI_SPARK = False)")

def _selisih(r):
    if r.get("length_new") is None:
        return None
    return r["length_new"] - (r.get("length_old") or 0)

def proses_batch(pesan):
    """List pesan Kafka -> list rekaman hasil (satu per pesan, valid atau tidak)."""
    baris = [(m.partition(), m.offset(), m.value().decode("utf-8", "replace")) for m in pesan]
    if PAKAI_SPARK:
        df = spark.createDataFrame(baris, SKEMA_MENTAH)
        hasil = (df.select("partition", "offset", F.from_json("value", SKEMA).alias("e"))
                   .select("partition", "offset", "e.*")
                   .withColumn("valid", F.col("id").isNotNull() & F.col("wiki").isNotNull())
                   .withColumn("selisih_byte", F.col("length_new") - F.coalesce(F.col("length_old"), F.lit(0))))
        return [r.asDict() for r in hasil.collect()]
    out = []
    for p, o, v in baris:
        try:
            e = json.loads(v)
        except ValueError:
            e = {}
        r = {"partition": p, "offset": o, **{k: e.get(k) for k in KOLOM}}
        r["valid"] = r["id"] is not None and r["wiki"] is not None
        r["selisih_byte"] = _selisih(r)
        out.append(r)
    return out

print("Fungsi pengolahan siap.")

Spark 4.1.3 | master: local[2] | UI: http://127.0.0.1:4042
Fungsi pengolahan siap.


## Sel 3. Loop Worker
Sel ini terus berjalan. Urutan tiap batch: **ambil → olah → kirim hasil → tunggu konfirmasi broker → commit offset**.
Karena commit dilakukan paling akhir, batch yang terputus di tengah akan diproses ulang saat Worker hidup lagi (at-least-once).

In [ ]:
# ===== SEL 3: Loop Worker - consumer group wiki-pekerja =====
class CrashSimulasi(Exception):
    pass

def saat_dapat_partisi(c, parts):
    awal = {}
    for tp in c.committed(parts, timeout=10):
        awal[tp.partition] = tp.offset if tp.offset >= 0 else "awal topic"
    print(f"[{time.strftime('%H:%M:%S')}] Dapat partisi {sorted(awal)} | melanjutkan dari offset ter-commit: {awal}")
    lapor("SIAP", partisi=sorted(awal), offset_mulai={str(k): v for k, v in awal.items()})
    produser.flush(5)

def saat_lepas_partisi(c, parts):
    print(f"[{time.strftime('%H:%M:%S')}] Partisi dilepas: {sorted(p.partition for p in parts)}")

konsumen = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": GROUP_ID,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,          # commit manual SETELAH hasil terkirim
    "max.poll.interval.ms": 300000,
    "session.timeout.ms": 45000,
})
konsumen.subscribe([TOPIC_RAW], on_assign=saat_dapat_partisi, on_revoke=saat_lepas_partisi)

total_pesan, total_batch, detak = 0, 0, 0
alasan = "Interrupt"
print(f"Worker {HOST} membaca '{TOPIC_RAW}' dalam group '{GROUP_ID}' ... (Interrupt untuk berhenti)")
try:
    while True:
        batch, t0 = [], time.time()
        while time.time() - t0 < BATCH_DETIK and len(batch) < BATCH_MAKS:
            for m in konsumen.consume(num_messages=BATCH_MAKS - len(batch), timeout=1.0):
                if m.error():
                    if m.error().code() != KafkaError._PARTITION_EOF:
                        print("Kafka:", m.error())
                    continue
                batch.append(m)
            if time.time() - detak >= DETAK_DETIK:
                lapor("HIDUP", pesan=total_pesan, batch=total_batch)
                detak = time.time()
        if not batch:
            continue

        p0 = time.perf_counter()
        rekaman = proses_batch(batch)                                   # 1. olah
        t_olah = time.perf_counter() - p0
        id_batch = f"{HOST}-{total_batch + 1}"
        galat_kirim.clear()
        for r in rekaman:                                               # 2. kirim hasil
            kunci = f"{r['wiki']}:{r['id']}" if r["valid"] else f"invalid:{r['partition']}:{r['offset']}"
            kirim(TOPIC_HASIL, kunci, {**r, "worker": HOST, "batch": id_batch, "t_proses": time.time()})
        sisa = produser.flush(30)                                       # 3. tunggu konfirmasi broker
        if sisa or galat_kirim:
            raise RuntimeError(f"hasil belum terkirim (sisa={sisa}, galat={galat_kirim[:1]}); offset TIDAK di-commit")

        if SIMULASI_CRASH_SEBELUM_COMMIT:
            raise CrashSimulasi(id_batch)

        konsumen.commit(asynchronous=False)                             # 4. commit offset
        total_batch += 1
        total_pesan += len(batch)
        rentang = {}
        for m in batch:
            lo, hi = rentang.get(m.partition(), (m.offset(), m.offset()))
            rentang[m.partition()] = (min(lo, m.offset()), max(hi, m.offset()))
        lapor("BATCH", batch=id_batch, jumlah=len(batch), rentang={str(k): v for k, v in rentang.items()})
        print(f"[{time.strftime('%H:%M:%S')}] {id_batch}: {len(batch):>5} pesan | olah {t_olah:.2f} dtk | "
              f"offset {dict(sorted(rentang.items()))} | commit OK")
except KeyboardInterrupt:
    alasan = "Interrupt"
    print("\nWorker dihentikan (Interrupt). Batch yang belum di-commit akan diproses ulang saat Worker hidup lagi.")
except CrashSimulasi as e:
    alasan = "crash simulasi"
    print(f"\nCRASH SIMULASI: hasil batch {e} sudah terkirim tetapi offset TIDAK di-commit.")
    print("Set SIMULASI_CRASH_SEBELUM_COMMIT = False di Sel 1, jalankan Sel 1 lalu Sel 3 lagi -> batch ini diproses ulang (duplikat).")
finally:
    lapor("BERHENTI", alasan=alasan, pesan=total_pesan, batch=total_batch)
    produser.flush(10)
    konsumen.close()                                                    # tanpa commit (auto commit mati)
    print(f"Ringkasan sesi: {total_batch} batch, {total_pesan} pesan di-commit.")

Worker NthaanSnn-w3 membaca 'wiki.recentchange.raw' dalam group 'wiki-pekerja' ... (Interrupt untuk berhenti)
[13:20:16] Dapat partisi [0, 1] | melanjutkan dari offset ter-commit: {0: 12914, 1: 13078}
[13:20:19] Partisi dilepas: [0, 1]
[13:20:19] Dapat partisi [0, 1] | melanjutkan dari offset ter-commit: {0: 12914, 1: 13078}
[13:20:23] NthaanSnn-w3-1:  1225 pesan | olah 1.56 dtk | offset {0: (12914, 13207), 1: (13078, 13411)} | commit OK
[13:20:31] Partisi dilepas: [0, 1]
[13:20:31] Dapat partisi [1] | melanjutkan dari offset ter-commit: {1: 13412}
[13:20:35] NthaanSnn-w3-2:   140 pesan | olah 1.58 dtk | offset {0: (13208, 13249), 1: (13412, 13466)} | commit OK
[13:20:46] NthaanSnn-w3-3:    47 pesan | olah 1.54 dtk | offset {1: (13467, 13513)} | commit OK
[13:20:58] NthaanSnn-w3-4:    43 pesan | olah 1.54 dtk | offset {1: (13514, 13556)} | commit OK
[13:21:10] NthaanSnn-w3-5:    54 pesan | olah 1.55 dtk | offset {1: (13557, 13610)} | commit OK
[13:21:21] NthaanSnn-w3-6:    69 pesan | o